# Laboratoire de recommandation JaimeTV

**Phase 1 :** explorer les métadonnées TMDB locales et établir une référence TF-IDF. La base SQLite est lue en mode seule lecture; aucun service externe ni export n’est utilisé. Le notebook s’exécute sur l’ordinateur local, pas sur la VM Oracle.

Le corpus utilise le titre, le résumé espagnol et les genres TMDB. TF-IDF classe les titres par mots communs; il ne comprend pas encore les filtres exacts ni les préférences subjectives. Les dix prompts restent en espagnol, langue prévue pour l’application.

## Plan et validations

1. **Environnement :** Python 3.12+; installer localement `requirements.txt` et `requirements-lab.txt`.
2. **Critères :** métadonnées actuelles, matrice TF-IDF valide, résultats uniques (maximum cinq) et scores finis. La pertinence humaine reste à évaluer.
3. **Structure :** réutiliser `catalog.fingerprint`; lire la base sans la modifier.
4. **Phase 1, MVP :** métadonnées TMDB → TF-IDF → classement de titres.
5. **Tests :** assertions et tests de fumée sur les dix prompts.
6. **Phase 2 :** modèle sémantique espagnol et filtres exacts; attendre la validation de la phase 1.
7. **Mesures :** comparer au baseline; mesurer qualité, latence et mémoire.
8. **Robustesse :** traiter les entrées non prises en charge et les erreurs sans interrompre le catalogue.
9. **Avancement :** le checklist distingue validation technique et revue humaine. Les phases 2 et 3 restent bloquées jusqu’à validation.

In [15]:
import json
import os
import random
import sqlite3
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((path for path in candidates if (path / "catalog.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Ouvrez ce notebook dans le workspace JaimeTV pour trouver catalog.py.")

configured_path = os.environ.get("IPTV_DB_PATH")
DATABASE_PATH = Path(configured_path).expanduser() if configured_path else PROJECT_ROOT / "catalog.db"
DATABASE_PATH = DATABASE_PATH.resolve()
if not DATABASE_PATH.is_file():
    raise FileNotFoundError(
        f"Base locale introuvable : {DATABASE_PATH}. Définissez IPTV_DB_PATH ou placez catalog.db dans le projet."
    )

sys.path.insert(0, str(PROJECT_ROOT))
from catalog import fingerprint

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        "scikit-learn manque dans cet environnement. Installez-le avec : "
        "python -m pip install -r requirements.txt -r requirements-lab.txt"
    ) from error

size_mib = DATABASE_PATH.stat().st_size / (1024 ** 2)
print(f"Projet : {PROJECT_ROOT.name}")
print(f"Base utilisée : {DATABASE_PATH.name} ({size_mib:.1f} Mio), lecture seule.")
print(f"Graine reproductible : {SEED}")

Projet : iptv
Base utilisée : catalog.db (42.9 Mio), lecture seule.
Graine reproductible : 42


In [16]:
CATALOGS = {
    "movies": ("movies", "stream_id"),
    "series": ("series", "series_id"),
}
COUNTRY = "CA"
matched_records = []
stale_matched = {kind: 0 for kind in CATALOGS}
raw_totals = {}
status_counts = {}

connection = sqlite3.connect(f"{DATABASE_PATH.as_uri()}?mode=ro", uri=True)
connection.row_factory = sqlite3.Row
try:
    tables = {row[0] for row in connection.execute("SELECT name FROM sqlite_master WHERE type='table'")}
    required_tables = {"tmdb_metadata", *(table for table, _ in CATALOGS.values())}
    if not required_tables.issubset(tables):
        missing = sorted(required_tables - tables)
        raise RuntimeError(f"Tables requises absentes : {missing}")

    for kind, (table, key) in CATALOGS.items():
        raw_totals[kind] = connection.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        rows = connection.execute(
            f"""SELECT CAST(s.{key} AS TEXT) AS local_record_id, s.raw_json,
                       m.fingerprint, m.payload
                FROM {table} AS s
                JOIN tmdb_metadata AS m
                  ON m.kind = ? AND m.record_id = CAST(s.{key} AS TEXT) AND m.country = ?
                WHERE json_extract(m.payload, '$.status') = 'matched'""",
            (kind, COUNTRY),
        ).fetchall()

        for row in rows:
            raw = json.loads(row["raw_json"])
            if fingerprint(raw) != row["fingerprint"]:
                stale_matched[kind] += 1
                continue
            payload = json.loads(row["payload"])
            genres = payload.get("genres") if isinstance(payload.get("genres"), list) else []
            providers = payload.get("providers") if isinstance(payload.get("providers"), dict) else {}
            provider_names = sorted({
                str(name).strip()
                for names in providers.values() if isinstance(names, list)
                for name in names if isinstance(name, str) and name.strip()
            }, key=str.casefold)
            title = payload.get("title") if isinstance(payload.get("title"), str) else ""
            overview = payload.get("overview") if isinstance(payload.get("overview"), str) else ""
            genres = [name.strip() for name in genres if isinstance(name, str) and name.strip()]
            searchable_text = " ".join(part for part in (title, " ".join(genres), overview) if part).strip()
            matched_records.append({
                "kind": kind,
                "local_record_id": row["local_record_id"],
                "tmdb_id": payload.get("id"),
                "title": title,
                "overview": overview,
                "genres": genres,
                "providers": provider_names,
                "text": searchable_text,
            })

    for row in connection.execute("SELECT kind, payload FROM tmdb_metadata WHERE country=?", (COUNTRY,)):
        payload = json.loads(row["payload"])
        status = payload.get("status", "unknown")
        key = (row["kind"], status)
        status_counts[key] = status_counts.get(key, 0) + 1
finally:
    connection.close()

record_df = pd.DataFrame(matched_records)
if record_df.empty:
    catalog_df = pd.DataFrame(columns=[
        "kind", "tmdb_id", "local_record_ids", "title", "overview", "genres", "providers", "text"
    ])
else:
    usable_df = record_df[(record_df["tmdb_id"].notna()) & (record_df["title"].str.strip() != "")]
    catalog_rows = []
    for (kind, tmdb_id), group in usable_df.groupby(["kind", "tmdb_id"], sort=True):
        first = group.iloc[0]
        catalog_rows.append({
            "kind": kind,
            "tmdb_id": int(tmdb_id),
            "local_record_ids": sorted(group["local_record_id"].unique()),
            "title": first["title"],
            "overview": next((value for value in group["overview"] if value), ""),
            "genres": first["genres"],
            "providers": sorted({name for names in group["providers"] for name in names}, key=str.casefold),
            "text": next((value for value in group["text"] if value), first["title"]),
        })
    catalog_df = pd.DataFrame(catalog_rows)

kind_labels = {"movies": "Films", "series": "Séries"}
coverage_rows = []
for kind in CATALOGS:
    current_matches = sum(row["kind"] == kind for row in matched_records)
    unique_titles = int((catalog_df["kind"] == kind).sum()) if not catalog_df.empty else 0
    coverage_rows.append({
        "Type": kind_labels[kind],
        "Entrées locales": raw_totals.get(kind, 0),
        "Métadonnées TMDB valides": current_matches,
        "Caches périmées exclues": stale_matched[kind],
        "Titres TMDB uniques": unique_titles,
        "Copies regroupées": current_matches - unique_titles,
    })
coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)
status_labels = {
    "matched": "associé", "candidates": "candidats", "missing": "sans correspondance",
    "pending": "en attente", "unknown": "inconnu",
}
status_df = pd.DataFrame([
    {"Type": kind_labels.get(kind, kind), "Statut TMDB": status_labels.get(status, status), "Nombre": count}
    for (kind, status), count in sorted(status_counts.items())
])
display(status_df)
print(f"Titres TMDB uniques exploitables : {len(catalog_df)}")

,Type,Entrées locales,Métadonnées TMDB valides,Caches périmées exclues,Titres TMDB uniques,Copies regroupées
0,Films,6910,6864,0,6658,206
1,Séries,2368,2348,0,2305,43


,Type,Statut TMDB,Nombre
0,Films,candidats,6
1,Films,associé,6864
2,Films,sans correspondance,28
3,Séries,associé,2348
4,Séries,sans correspondance,10


Titres TMDB uniques exploitables : 8963


In [17]:
REFERENCE_TITLE = "Scary Movie"
reference_status = {
    "Entrées locales": 0,
    "Association TMDB valide": 0,
    "Autres statuts actuels": 0,
    "Cache périmé": 0,
    "Sans cache TMDB": 0,
}
connection = sqlite3.connect(f"{DATABASE_PATH.as_uri()}?mode=ro", uri=True)
connection.row_factory = sqlite3.Row
try:
    for kind, (table, key) in CATALOGS.items():
        rows = connection.execute(
            f"""SELECT s.raw_json, m.fingerprint, m.payload
                FROM {table} AS s
                LEFT JOIN tmdb_metadata AS m
                  ON m.kind=? AND m.record_id=CAST(s.{key} AS TEXT) AND m.country=?
                WHERE LOWER(s.name) LIKE ?""",
            (kind, COUNTRY, f"%{REFERENCE_TITLE.casefold()}%"),
        ).fetchall()
        for row in rows:
            reference_status["Entrées locales"] += 1
            if row["payload"] is None:
                reference_status["Sans cache TMDB"] += 1
                continue
            if fingerprint(json.loads(row["raw_json"])) != row["fingerprint"]:
                reference_status["Cache périmé"] += 1
                continue
            payload = json.loads(row["payload"])
            if payload.get("status") == "matched":
                reference_status["Association TMDB valide"] += 1
            else:
                reference_status["Autres statuts actuels"] += 1
finally:
    connection.close()

display(pd.DataFrame([reference_status], index=[REFERENCE_TITLE]))

,Entrées locales,Association TMDB valide,Autres statuts actuels,Cache périmé,Sans cache TMDB
Scary Movie,6,6,0,0,0


In [18]:
vectorizer = None
tfidf_matrix = None
baseline_ready = False

if not catalog_df.empty:
    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
        stop_words=None,  # Garde « no »; TF-IDF ne comprend toutefois pas la négation.
    )
    tfidf_matrix = vectorizer.fit_transform(catalog_df["text"].fillna(""))
    baseline_ready = tfidf_matrix.shape[0] == len(catalog_df) and tfidf_matrix.shape[1] > 0
    print(f"Matrice TF-IDF : {tfidf_matrix.shape[0]} titres × {tfidf_matrix.shape[1]} termes")
else:
    print("Aucun titre TMDB exploitable; matrice TF-IDF non créée.")

RESULT_COLUMNS = ["kind", "title", "genres", "score", "copies"]

def recommend_prompt(prompt, kind=None, top_n=5):
    """Baseline lexical : compare les mots du prompt aux métadonnées TMDB, sans filtres exacts."""
    if not baseline_ready:
        return pd.DataFrame(columns=RESULT_COLUMNS)
    query_vector = vectorizer.transform([str(prompt)])
    scores = cosine_similarity(query_vector, tfidf_matrix).ravel()
    candidates = np.arange(len(catalog_df))
    if kind is not None:
        candidates = candidates[catalog_df.iloc[candidates]["kind"].to_numpy() == kind]
    if not len(candidates):
        return pd.DataFrame(columns=RESULT_COLUMNS)
    ordered = candidates[np.argsort(-scores[candidates], kind="stable")[:max(0, top_n)]]
    rows = catalog_df.iloc[ordered]
    return pd.DataFrame({
        "kind": rows["kind"].to_numpy(),
        "title": rows["title"].to_numpy(),
        "genres": [", ".join(value) for value in rows["genres"]],
        "score": scores[ordered],
        "copies": [len(value) for value in rows["local_record_ids"]],
    }).reset_index(drop=True)


def recommend_similar_to_title(reference_title, kind=None, top_n=5):
    """Compare les métadonnées d’un titre de référence aux autres; correspondance exacte pour cette phase."""
    if not baseline_ready:
        return pd.DataFrame(columns=RESULT_COLUMNS)
    matches = catalog_df["title"].str.casefold() == str(reference_title).strip().casefold()
    if kind is not None:
        matches &= catalog_df["kind"] == kind
    reference_indices = np.flatnonzero(matches.to_numpy())
    if not len(reference_indices):
        return pd.DataFrame(columns=RESULT_COLUMNS)
    reference_index = reference_indices[0]
    scores = cosine_similarity(tfidf_matrix[reference_index], tfidf_matrix).ravel()
    candidates = np.arange(len(catalog_df))
    candidates = candidates[candidates != reference_index]
    if kind is not None:
        candidates = candidates[catalog_df.iloc[candidates]["kind"].to_numpy() == kind]
    ordered = candidates[np.argsort(-scores[candidates], kind="stable")[:max(0, top_n)]]
    rows = catalog_df.iloc[ordered]
    return pd.DataFrame({
        "kind": rows["kind"].to_numpy(),
        "title": rows["title"].to_numpy(),
        "genres": [", ".join(value) for value in rows["genres"]],
        "score": scores[ordered],
        "copies": [len(value) for value in rows["local_record_ids"]],
    }).reset_index(drop=True)

print("Baseline prêt :", baseline_ready)
print("Les scores classent les résultats; ils ne mesurent pas une pertinence calibrée.")

Matrice TF-IDF : 8963 titres × 250805 termes
Baseline prêt : True
Les scores classent les résultats; ils ne mesurent pas une pertinence calibrée.


### Phase 2 — modèle sémantique préentraîné

Candidat local : `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (espagnol pris en charge, vecteurs de 384 dimensions, licence Apache-2.0, environ 100 M de paramètres). On ne l’entraîne pas : on compare la similarité de sens au baseline TF-IDF.

Premier essai sur CPU, uniquement dans le notebook local. Les poids sont téléchargés dans le cache Hugging Face, pas dans le dépôt. On mesurera chargement, encodage du catalogue, latence des prompts et mémoire du processus. Cette mesure sur Mac ne garantit pas que le modèle tiendra dans la VM Oracle de 1 Gio.

In [19]:
import gc
import time
import psutil
from sentence_transformers import SentenceTransformer

SEMANTIC_MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
process = psutil.Process()
semantic_model = None
semantic_embeddings = None
semantic_ready = False
semantic_error = None
semantic_metrics = {}

if catalog_df.empty:
    semantic_error = "Aucun titre TMDB exploitable dans la base."
    print(semantic_error)
else:
    try:
        gc.collect()
        rss_before_mib = process.memory_info().rss / (1024 ** 2)
        load_started = time.perf_counter()
        semantic_model = SentenceTransformer(SEMANTIC_MODEL_ID, device="cpu")
        model_load_seconds = time.perf_counter() - load_started
        rss_after_load_mib = process.memory_info().rss / (1024 ** 2)

        encoding_started = time.perf_counter()
        semantic_embeddings = semantic_model.encode(
            catalog_df["text"].fillna("").astype(str).tolist(),
            batch_size=16,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        catalog_encoding_seconds = time.perf_counter() - encoding_started
        rss_after_encoding_mib = process.memory_info().rss / (1024 ** 2)
        semantic_ready = (
            semantic_embeddings.shape == (len(catalog_df), 384)
            and np.isfinite(semantic_embeddings).all()
        )
        semantic_metrics = {
            "Modèle": SEMANTIC_MODEL_ID,
            "Appareil": "CPU",
            "Titres encodés": len(catalog_df),
            "Dimensions": semantic_embeddings.shape[1],
            "Chargement (s)": round(model_load_seconds, 2),
            "Encodage du catalogue (s)": round(catalog_encoding_seconds, 2),
            "RAM avant (Mio)": round(rss_before_mib, 1),
            "RAM après chargement (Mio)": round(rss_after_load_mib, 1),
            "RAM après encodage (Mio)": round(rss_after_encoding_mib, 1),
        }
        display(pd.DataFrame([semantic_metrics]))
        print("Modèle sémantique prêt :", semantic_ready)
    except Exception as error:
        semantic_error = f"{type(error).__name__}: {error}"
        semantic_model = None
        semantic_embeddings = None
        semantic_ready = False
        print("Le modèle sémantique n’a pas pu être chargé/exécuté; TF-IDF reste disponible.")
        print(semantic_error[:500])

Batches: 100%|██████████| 561/561 [00:30<00:00, 18.19it/s]


,Modèle,Appareil,Titres encodés,Dimensions,Chargement (s),Encodage du catalogue (s),RAM avant (Mio),RAM après chargement (Mio),RAM après encodage (Mio)
0,sentence-transformers/paraphrase-multilingual-...,CPU,8963,384,1.37,30.85,879.0,1082.5,1414.6


Modèle sémantique prêt : True


In [20]:
SEMANTIC_RESULT_COLUMNS = ["kind", "title", "genres", "score", "copies"]

def recommend_semantic_prompt(prompt, kind=None, top_n=5):
    """Classe le catalogue par similarité cosinus des embeddings normalisés."""
    if not semantic_ready:
        return pd.DataFrame(columns=SEMANTIC_RESULT_COLUMNS)
    query_vector = semantic_model.encode(
        [str(prompt)], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False
    )[0]
    scores = semantic_embeddings @ query_vector
    candidates = np.arange(len(catalog_df))
    if kind is not None:
        candidates = candidates[catalog_df.iloc[candidates]["kind"].to_numpy() == kind]
    ordered = candidates[np.argsort(-scores[candidates], kind="stable")[:max(0, top_n)]]
    rows = catalog_df.iloc[ordered]
    return pd.DataFrame({
        "kind": rows["kind"].to_numpy(),
        "title": rows["title"].to_numpy(),
        "genres": [", ".join(value) for value in rows["genres"]],
        "score": scores[ordered],
        "copies": [len(value) for value in rows["local_record_ids"]],
    }).reset_index(drop=True)


def recommend_semantic_similar_to_title(reference_title, kind=None, top_n=5):
    """Classe les titres selon leur similarité avec un titre TMDB nommé explicitement."""
    if not semantic_ready:
        return pd.DataFrame(columns=SEMANTIC_RESULT_COLUMNS)
    matches = catalog_df["title"].str.casefold() == str(reference_title).strip().casefold()
    if kind is not None:
        matches &= catalog_df["kind"] == kind
    reference_indices = np.flatnonzero(matches.to_numpy())
    if not len(reference_indices):
        return pd.DataFrame(columns=SEMANTIC_RESULT_COLUMNS)
    reference_index = reference_indices[0]
    scores = semantic_embeddings @ semantic_embeddings[reference_index]
    candidates = np.arange(len(catalog_df))
    candidates = candidates[candidates != reference_index]
    if kind is not None:
        candidates = candidates[catalog_df.iloc[candidates]["kind"].to_numpy() == kind]
    ordered = candidates[np.argsort(-scores[candidates], kind="stable")[:max(0, top_n)]]
    rows = catalog_df.iloc[ordered]
    return pd.DataFrame({
        "kind": rows["kind"].to_numpy(),
        "title": rows["title"].to_numpy(),
        "genres": [", ".join(value) for value in rows["genres"]],
        "score": scores[ordered],
        "copies": [len(value) for value in rows["local_record_ids"]],
    }).reset_index(drop=True)

print("Recherche sémantique prête :", semantic_ready)

Recherche sémantique prête : True


### Dix prompts d’évaluation

Ces exemples servent à évaluer, pas à entraîner TF-IDF. Le baseline ne comprend pas « seulement », les plateformes, la négation ni les critères non pris en charge. Les prompts restent en espagnol, langue de l’application.

1. « Quiero solo series de ciencia ficción. »
2. « Recomiéndame películas de comedia disponibles en Netflix. »
3. « Busco algo de ciencia ficción disponible en Netflix. »
4. « Recomiéndame algo parecido a Scary Movie. »
5. « Quiero algo parecido a Scary Movie, pero no muy violento. »
6. « Quiero una película de misterio con un ambiente oscuro. »
7. « Dame una comedia de menos de 90 minutos. »
8. « Solo series estrenadas después de 2020. »
9. « Busco series de ciencia ficción disponibles en CineNube. »
10. « Recomiéndame algo parecido a La nebulosa de los lunes 9843. »

Réviser jusqu’à cinq titres par prompt. *Scary Movie* est le cas ambigu : la comédie et les mentions de violence ne permettent pas de conclure sur son niveau réel.

In [21]:
TEST_PROMPTS = [
    "Quiero solo series de ciencia ficción.",
    "Recomiéndame películas de comedia disponibles en Netflix.",
    "Busco algo de ciencia ficción disponible en Netflix.",
    "Recomiéndame algo parecido a Scary Movie.",
    "Quiero algo parecido a Scary Movie, pero no muy violento.",
    "Quiero una película de misterio con un ambiente oscuro.",
    "Dame una comedia de menos de 90 minutos.",
    "Solo series estrenadas después de 2020.",
    "Busco series de ciencia ficción disponibles en CineNube.",
    "Recomiéndame algo parecido a La nebulosa de los lunes 9843.",
]

prediction_rows = []
for prompt_id, prompt in enumerate(TEST_PROMPTS, start=1):
    results = recommend_prompt(prompt, top_n=5)
    for rank, result in results.iterrows():
        prediction_rows.append({
            "prompt_id": prompt_id,
            "prompt": prompt,
            "rank": rank + 1,
            "tipo": result["kind"],
            "titulo": result["title"],
            "generos": result["genres"],
            "score_tfidf": round(float(result["score"]), 4),
            "revision_humana": "",
            "notas": "",
        })

review_df = pd.DataFrame(prediction_rows, columns=[
    "prompt_id", "prompt", "rank", "tipo", "titulo", "generos",
    "score_tfidf", "revision_humana", "notas",
])
print(f"Candidats à examiner : {len(review_df)} (maximum 50)")
print("Étiquettes : pertinente / incertaine / non pertinente.")
review_columns = {
    "prompt_id": "ID", "prompt": "Prompt (espagnol)", "rank": "Rang",
    "tipo": "Type", "titulo": "Titre TMDB", "generos": "Genres TMDB (espagnol)",
    "score_tfidf": "Score TF-IDF", "revision_humana": "Évaluation humaine", "notas": "Notes",
}
review_display = review_df.rename(columns=review_columns).copy()
review_display["Type"] = review_display["Type"].map({"movies": "Films", "series": "Séries"})
display(review_display)

scary_exists = (
    not catalog_df.empty
    and catalog_df["title"].str.casefold().eq("scary movie").any()
)
if scary_exists:
    print("Titres similaires à Scary Movie selon ses métadonnées TMDB :")
    scary_results = recommend_similar_to_title("Scary Movie", top_n=5).rename(columns={
        "kind": "Type", "title": "Titre TMDB", "genres": "Genres TMDB (espagnol)",
        "score": "Score TF-IDF", "copies": "Copies locales",
    })
    scary_results["Type"] = scary_results["Type"].map({"movies": "Films", "series": "Séries"})
    display(scary_results)
else:
    print(
        "Scary Movie existe dans le catalogue, mais sans cache TMDB actuel. "
        "Consultez ce titre dans l’administration, puis relancez les cellules d’inventaire et de recommandation."
    )

Candidats à examiner : 50 (maximum 50)
Étiquettes : pertinente / incertaine / non pertinente.


,ID,Prompt (espagnol),Rang,Type,Titre TMDB,Genres TMDB (espagnol),Score TF-IDF,Évaluation humaine,Notes
0,1,Quiero solo series de ciencia ficción.,1,Films,2.0,"Acción, Ciencia ficción, Fantasía",0.1967,,
1,1,Quiero solo series de ciencia ficción.,2,Films,"Invasion, U.S.A.","Drama, Ciencia ficción",0.1604,,
2,1,Quiero solo series de ciencia ficción.,3,Films,Inmortales,"Drama, Ciencia ficción",0.1556,,
3,1,Quiero solo series de ciencia ficción.,4,Films,Sentinel,"Ciencia ficción, Acción",0.1468,,
4,1,Quiero solo series de ciencia ficción.,5,Films,La promesa,"Drama, Ciencia ficción",0.1422,,
5,2,Recomiéndame películas de comedia disponibles ...,1,Séries,Cobra Kai,"Action & Adventure, Drama, Comedia",0.1034,,
6,2,Recomiéndame películas de comedia disponibles ...,2,Séries,Fast & Furious: Espías a todo gas,"Animación, Action & Adventure, Kids, Familia",0.0860,,
7,2,Recomiéndame películas de comedia disponibles ...,3,Films,David Holmes: El chico que sobrevivió,Documental,0.0851,,
8,2,Recomiéndame películas de comedia disponibles ...,4,Films,Val,Documental,0.0802,,
9,2,Recomiéndame películas de comedia disponibles ...,5,Séries,El Superagente 86,"Comedia, Sci-Fi & Fantasy",0.0790,,


Titres similaires à Scary Movie selon ses métadonnées TMDB :


,Type,Titre TMDB,Genres TMDB (espagnol),Score TF-IDF,Copies locales
0,Films,Scary Movie 2,Comedia,0.097978,1
1,Films,Scary Movie: Terroríficamente Incorrecta,Comedia,0.089514,1
2,Films,The Nan Movie,Comedia,0.088667,1
3,Films,Scary Movie 3,Comedia,0.079044,1
4,Séries,Prefiero la muerte,"Action & Adventure, Comedia",0.077625,1


### Phase 2 — comparaison sémantique et contraintes de déploiement

Le modèle préentraîné est testé sur Mac, sans entraînement. En FP32, il dépasse la RAM de la VM de 1 Gio. Une variante ONNX INT8 préconvertie réduit le poids à environ 113 Mio; l’indexation complète prend environ 4 minutes sur ARM avec un seul thread et le processus atteint environ 804 Mio de RAM. Ces mesures sont indicatives : la VM x86_64 doit encore être testée avec la variante AVX2.

La revue humaine a été volontairement sautée. La similarité et les scores ne prouvent donc pas que les recommandations sont précises. Cette phase compare les sorties et les ressources; elle ne met pas encore le modèle dans Flask.

In [22]:
comparison_rows = []
semantic_query_latencies = []
tfidf_query_latencies = []

for prompt_id, prompt in enumerate(TEST_PROMPTS, start=1):
    started = time.perf_counter()
    tfidf_results = recommend_prompt(prompt, top_n=5)
    tfidf_query_latencies.append((time.perf_counter() - started) * 1000)

    started = time.perf_counter()
    semantic_results = recommend_semantic_prompt(prompt, top_n=5)
    semantic_query_latencies.append((time.perf_counter() - started) * 1000)

    tfidf_titles = tfidf_results["title"].tolist()
    semantic_titles = semantic_results["title"].tolist()
    comparison_rows.append({
        "ID": prompt_id,
        "Prompt (espagnol)": prompt,
        "Top-5 TF-IDF": "\n".join(tfidf_titles),
        "Top-5 sémantique": "\n".join(semantic_titles) if semantic_titles else "Modèle indisponible",
        "Titres communs": len(set(tfidf_titles) & set(semantic_titles)),
        "TF-IDF (ms)": round(tfidf_query_latencies[-1], 1),
        "Sémantique (ms)": round(semantic_query_latencies[-1], 1),
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

if semantic_ready:
    semantic_metrics["Latence moyenne par prompt (ms)"] = round(float(np.mean(semantic_query_latencies)), 1)
    semantic_metrics["Latence P95 par prompt (ms)"] = round(float(np.percentile(semantic_query_latencies, 95)), 1)
    semantic_metrics["Latence moyenne TF-IDF (ms)"] = round(float(np.mean(tfidf_query_latencies)), 1)
    display(pd.DataFrame([semantic_metrics]))

if scary_exists and semantic_ready:
    print("Titres similaires à Scary Movie : TF-IDF vs. modèle sémantique")
    tfidf_scary = recommend_similar_to_title("Scary Movie", top_n=5)
    semantic_scary = recommend_semantic_similar_to_title("Scary Movie", top_n=5)
    scary_comparison = pd.DataFrame({
        "Rang": range(1, 6),
        "TF-IDF": tfidf_scary["title"].tolist(),
        "Sémantique": semantic_scary["title"].tolist(),
    })
    display(scary_comparison)

print("Sans étiquettes humaines, ces résultats ne permettent pas de mesurer la précision des recommandations.")

,ID,Prompt (espagnol),Top-5 TF-IDF,Top-5 sémantique,Titres communs,TF-IDF (ms),Sémantique (ms)
0,1,Quiero solo series de ciencia ficción.,"2.0\nInvasion, U.S.A.\nInmortales\nSentinel\nL...",2.0\nRobotech\nLa promesa\nRéplicas\nגחליליות,2,5.7,10.2
1,2,Recomiéndame películas de comedia disponibles ...,Cobra Kai\nFast & Furious: Espías a todo gas\n...,Quédate en casa con Goofy\nLa loca risa en vac...,0,5.2,10.3
2,3,Busco algo de ciencia ficción disponible en Ne...,"2.0\nInvasion, U.S.A.\nInmortales\nSentinel\nL...",2.0\nValle de los Dioses\nBobcat Goldthwait's ...,1,5.6,9.8
3,4,Recomiéndame algo parecido a Scary Movie.,Scary Movie: Terroríficamente Incorrecta\nScar...,Naturaleza de pesadilla\nScary Movie: Terroríf...,1,4.7,9.5
4,5,"Quiero algo parecido a Scary Movie, pero no mu...",Scary Movie: Terroríficamente Incorrecta\nScar...,Scary Movie: Terroríficamente Incorrecta\nCrón...,1,4.8,10.1
5,6,Quiero una película de misterio con un ambient...,"Salcedo, cuero y boogaloo\nオリエント急行殺人事件\nHandso...",オリエント急行殺人事件\nCiudad invisible\nMisterios sin r...,1,5.6,9.4
6,7,Dame una comedia de menos de 90 minutos.,El Hijo Perdido\nEl juego de la verdad\nKate\n...,Fin de semana\nEl Hijo Perdido\nLa risa en vac...,1,5.2,9.4
7,8,Solo series estrenadas después de 2020.,Unprecedented\nEddington\nThe First Wave\nMusi...,2026\nYou Only Live Twice\nEl director desnudo...,0,5.3,9.2
8,9,Busco series de ciencia ficción disponibles en...,"2.0\nInvasion, U.S.A.\nInmortales\nSentinel\nL...",Freaks of Nature\nValle de los Dioses\nDestroy...,1,5.0,9.5
9,10,Recomiéndame algo parecido a La nebulosa de lo...,Garfield: La película\nHannah Montana: La pelí...,Une minute de soleil en moins\n2067\n2026\nEl ...,0,5.5,9.7


,Modèle,Appareil,Titres encodés,Dimensions,Chargement (s),Encodage du catalogue (s),RAM avant (Mio),RAM après chargement (Mio),RAM après encodage (Mio),Latence moyenne par prompt (ms),Latence P95 par prompt (ms),Latence moyenne TF-IDF (ms)
0,sentence-transformers/paraphrase-multilingual-...,CPU,8963,384,1.37,30.85,879.0,1082.5,1414.6,9.7,10.3,5.3


Titres similaires à Scary Movie : TF-IDF vs. modèle sémantique


,Rang,TF-IDF,Sémantique
0,1,Scary Movie 2,Asesinato para principiantes
1,2,Scary Movie: Terroríficamente Incorrecta,El juicio
2,3,The Nan Movie,Clase letal
3,4,Scary Movie 3,Frank and Cindy
4,5,Prefiero la muerte,Depravación


Sans étiquettes humaines, ces résultats ne permettent pas de mesurer la précision des recommandations.


### Variante expérimentale : quantification dynamique INT8

Le modèle FP32 dépasse la RAM de la VM cible. Testons une quantification dynamique INT8 des couches linéaires, uniquement sur le Mac. On comparera mémoire, temps d’encodage et classement; cette variante ne sera pas déployée sans test sur la VM.

In [29]:
import gc
import hashlib
import platform
import time
from pathlib import Path

ONNX_MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
architecture = platform.machine().lower()
if architecture in {"arm64", "aarch64"}:
    ONNX_MODEL_FILE = "onnx/model_qint8_arm64.onnx"
elif architecture in {"x86_64", "amd64"}:
    # Oracle VM: x86_64 E2, compatible avec le graphe ONNX AVX2.
    ONNX_MODEL_FILE = "onnx/model_quint8_avx2.onnx"
else:
    ONNX_MODEL_FILE = None

onnx_ready = False
onnx_error = None
onnx_session = None
onnx_tokenizer = None
onnx_embeddings = None
onnx_metrics = {}

try:
    import onnxruntime as ort
    from huggingface_hub import hf_hub_download
    from transformers import AutoTokenizer
except ModuleNotFoundError as error:
    onnx_error = f"{type(error).__name__}: {error}"
    print("ONNX Runtime n’est pas disponible dans ce kernel.")
    print("Pour le test ONNX, choisissez le kernel Python 3.12 (requirements-lab.txt).")
else:
    if ONNX_MODEL_FILE is None:
        onnx_error = f"Architecture non prise en charge : {architecture}"
        print(onnx_error)
    else:
        try:
            # Le classement FP32 a déjà été affiché; libérer ce modèle avant ONNX.
            if "semantic_model" in globals():
                del semantic_model
            if "semantic_embeddings" in globals():
                del semantic_embeddings
            semantic_ready = False
            gc.collect()

            process = psutil.Process()
            rss_before_mib = process.memory_info().rss / (1024 ** 2)
            model_download_started = time.perf_counter()
            onnx_model_path = Path(hf_hub_download(
                repo_id=ONNX_MODEL_ID,
                filename=ONNX_MODEL_FILE,
            ))
            model_download_seconds = time.perf_counter() - model_download_started
            model_sha256 = hashlib.sha256(onnx_model_path.read_bytes()).hexdigest()

            options = ort.SessionOptions()
            options.intra_op_num_threads = 1
            options.inter_op_num_threads = 1
            options.enable_cpu_mem_arena = False
            options.enable_mem_pattern = False
            model_load_started = time.perf_counter()
            onnx_session = ort.InferenceSession(
                str(onnx_model_path),
                sess_options=options,
                providers=["CPUExecutionProvider"],
            )
            model_load_seconds = time.perf_counter() - model_load_started
            onnx_tokenizer = AutoTokenizer.from_pretrained(ONNX_MODEL_ID)
            onnx_input_names = {item.name for item in onnx_session.get_inputs()}
            rss_after_load_mib = process.memory_info().rss / (1024 ** 2)

            def encode_onnx(texts, batch_size=16):
                vectors = []
                for start in range(0, len(texts), batch_size):
                    batch = [str(text) for text in texts[start:start + batch_size]]
                    tokenized = onnx_tokenizer(
                        batch, padding=True, truncation=True, max_length=128,
                        return_tensors="np", return_token_type_ids=True,
                    )
                    feeds = {
                        name: value for name, value in tokenized.items()
                        if name in onnx_input_names
                    }
                    token_embeddings = onnx_session.run(None, feeds)[0]
                    mask = tokenized["attention_mask"].astype(np.float32)[..., None]
                    pooled = (token_embeddings * mask).sum(axis=1) / np.clip(mask.sum(axis=1), 1e-9, None)
                    pooled /= np.clip(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-12, None)
                    vectors.append(pooled.astype(np.float32))
                return np.vstack(vectors) if vectors else np.empty((0, 384), dtype=np.float32)

            index_started = time.perf_counter()
            onnx_embeddings = encode_onnx(catalog_df["text"].fillna("").tolist())
            index_seconds = time.perf_counter() - index_started
            rss_after_index_mib = process.memory_info().rss / (1024 ** 2)
            onnx_ready = (
                onnx_embeddings.shape == (len(catalog_df), 384)
                and np.isfinite(onnx_embeddings).all()
            )
            onnx_metrics = {
                "Architecture": architecture,
                "Artefact ONNX": ONNX_MODEL_FILE,
                "Taille du modèle (Mio)": round(onnx_model_path.stat().st_size / (1024 ** 2), 1),
                "SHA-256 (préfixe)": model_sha256[:16],
                "Titres encodés": len(catalog_df),
                "Dimensions": onnx_embeddings.shape[1],
                "Téléchargement/cache (s)": round(model_download_seconds, 2),
                "Chargement ONNX (s)": round(model_load_seconds, 2),
                "Indexation CPU, 1 thread (s)": round(index_seconds, 2),
                "RAM avant (Mio)": round(rss_before_mib, 1),
                "RAM après chargement (Mio)": round(rss_after_load_mib, 1),
                "RAM après indexation (Mio)": round(rss_after_index_mib, 1),
            }
            display(pd.DataFrame([onnx_metrics]))
            print("Artefact ONNX prêt :", onnx_ready)
        except Exception as error:
            onnx_error = f"{type(error).__name__}: {error}"
            onnx_ready = False
            onnx_session = None
            onnx_embeddings = None
            print("L’expérience ONNX a échoué; TF-IDF reste disponible.")
            print(onnx_error[:500])

ONNX Runtime n’est pas disponible dans ce kernel.
Pour le test ONNX, choisissez le kernel Python 3.12 (requirements-lab.txt).


In [24]:
ONNX_RESULT_COLUMNS = ["kind", "title", "genres", "score", "copies"]

def recommend_onnx_prompt(prompt, kind=None, top_n=5):
    """Classe les titres par similarité cosinus des embeddings ONNX INT8 normalisés."""
    if not onnx_ready:
        return pd.DataFrame(columns=ONNX_RESULT_COLUMNS)
    query_vector = encode_onnx([str(prompt)], batch_size=1)[0]
    scores = onnx_embeddings @ query_vector
    candidates = np.arange(len(catalog_df))
    if kind is not None:
        candidates = candidates[catalog_df.iloc[candidates]["kind"].to_numpy() == kind]
    ordered = candidates[np.argsort(-scores[candidates], kind="stable")[:max(0, top_n)]]
    rows = catalog_df.iloc[ordered]
    return pd.DataFrame({
        "kind": rows["kind"].to_numpy(),
        "title": rows["title"].to_numpy(),
        "genres": [", ".join(value) for value in rows["genres"]],
        "score": scores[ordered],
        "copies": [len(value) for value in rows["local_record_ids"]],
    }).reset_index(drop=True)


def recommend_onnx_similar_to_title(reference_title, kind=None, top_n=5):
    """Classe les titres par proximité ONNX avec une référence TMDB exacte."""
    if not onnx_ready:
        return pd.DataFrame(columns=ONNX_RESULT_COLUMNS)
    matches = catalog_df["title"].str.casefold() == str(reference_title).strip().casefold()
    if kind is not None:
        matches &= catalog_df["kind"] == kind
    reference_indices = np.flatnonzero(matches.to_numpy())
    if not len(reference_indices):
        return pd.DataFrame(columns=ONNX_RESULT_COLUMNS)
    reference_index = reference_indices[0]
    scores = onnx_embeddings @ onnx_embeddings[reference_index]
    candidates = np.arange(len(catalog_df))
    candidates = candidates[candidates != reference_index]
    if kind is not None:
        candidates = candidates[catalog_df.iloc[candidates]["kind"].to_numpy() == kind]
    ordered = candidates[np.argsort(-scores[candidates], kind="stable")[:max(0, top_n)]]
    rows = catalog_df.iloc[ordered]
    return pd.DataFrame({
        "kind": rows["kind"].to_numpy(),
        "title": rows["title"].to_numpy(),
        "genres": [", ".join(value) for value in rows["genres"]],
        "score": scores[ordered],
        "copies": [len(value) for value in rows["local_record_ids"]],
    }).reset_index(drop=True)

print("Recherche ONNX INT8 prête :", onnx_ready)

Recherche ONNX INT8 prête : False


In [25]:
onnx_comparison_rows = []
onnx_query_latencies = []

if onnx_ready:
    for prompt_id, prompt in enumerate(TEST_PROMPTS, start=1):
        started = time.perf_counter()
        tfidf_results = recommend_prompt(prompt, top_n=5)
        tfidf_titles = tfidf_results["title"].tolist()

        started = time.perf_counter()
        onnx_results = recommend_onnx_prompt(prompt, top_n=5)
        onnx_query_latencies.append((time.perf_counter() - started) * 1000)
        onnx_titles = onnx_results["title"].tolist()

        onnx_comparison_rows.append({
            "ID": prompt_id,
            "Prompt (espagnol)": prompt,
            "Top-5 TF-IDF": "\n".join(tfidf_titles),
            "Top-5 ONNX INT8": "\n".join(onnx_titles),
            "Titres communs": len(set(tfidf_titles) & set(onnx_titles)),
            "Latence ONNX (ms)": round(onnx_query_latencies[-1], 1),
        })

    onnx_comparison_df = pd.DataFrame(onnx_comparison_rows)
    display(onnx_comparison_df)
    onnx_metrics["Latence moyenne par prompt (ms)"] = round(float(np.mean(onnx_query_latencies)), 1)
    onnx_metrics["Latence P95 par prompt (ms)"] = round(float(np.percentile(onnx_query_latencies, 95)), 1)
    display(pd.DataFrame([onnx_metrics]))

    onnx_scary = recommend_onnx_similar_to_title("Scary Movie", top_n=5)
    print("Similarité à Scary Movie avec ONNX INT8 :")
    display(onnx_scary.rename(columns={
        "kind": "Type", "title": "Titre TMDB", "genres": "Genres TMDB (espagnol)",
        "score": "Score cosinus", "copies": "Copies locales",
    }))
    print("Déploiement Oracle non validé : le Mac est ARM; la VM Oracle utilisera la variante x86 AVX2.")
else:
    print("Comparaison ONNX non exécutée dans ce kernel.")
    print(onnx_error or "Choisissez Python 3.12 avec ONNX Runtime pour continuer.")

Comparaison ONNX non exécutée dans ce kernel.
ModuleNotFoundError: No module named 'onnxruntime'


### Étiquettes de revue

- **pertinente** : les métadonnées soutiennent la recommandation.
- **incertaine** : signaux contradictoires ou données insuffisantes.
- **non pertinente** : les métadonnées ne soutiennent pas la recommandation.

Les scores TF-IDF servent uniquement au classement; ce ne sont ni des probabilités ni des mesures calibrées. La qualité reste à valider manuellement.

In [26]:
# Copie ici les lignes à évaluer, en reprenant ID, Type et Titre du tableau ci-dessus.
# Exemple : (1, "Films", "2.0"): "non pertinente"
HUMAN_LABELS = {
    # (prompt_id, "Films" ou "Séries", "titre exact") : "pertinente / incertaine / non pertinente",
}
ALLOWED_LABELS = {"pertinente", "incertaine", "non pertinente"}
TYPE_LABELS = {"movies": "Films", "series": "Séries"}

if not review_df.empty:
    review_df["revision_humana"] = [
        HUMAN_LABELS.get((int(row.prompt_id), TYPE_LABELS.get(row.tipo, row.tipo), row.titulo), "")
        for row in review_df.itertuples(index=False)
    ]
invalid_labels = set(review_df["revision_humana"].dropna()) - {"", *ALLOWED_LABELS}
if invalid_labels:
    raise ValueError(f"Étiquettes inconnues : {sorted(invalid_labels)}")

reviewed_count = int(review_df["revision_humana"].isin(ALLOWED_LABELS).sum()) if not review_df.empty else 0
print(f"Résultats évalués : {reviewed_count} sur {len(review_df)}")
review_columns = {
    "prompt_id": "ID", "prompt": "Prompt (espagnol)", "rank": "Rang",
    "tipo": "Type", "titulo": "Titre", "generos": "Genres",
    "score_tfidf": "Score TF-IDF", "revision_humana": "Évaluation humaine", "notas": "Notes",
}
display(review_df[review_df["revision_humana"] != ""].rename(columns=review_columns))

Résultats évalués : 0 sur 50


,ID,Prompt (espagnol),Rang,Type,Titre,Genres,Score TF-IDF,Évaluation humaine,Notes


In [ ]:
smoke_results = recommend_prompt("comedia de ciencia ficción", top_n=5)
smoke_scores_finite = (
    bool(np.isfinite(smoke_results["score"].to_numpy()).all()) if not smoke_results.empty else False
)
smoke_titles_unique = (
    not smoke_results["title"].duplicated().any() if not smoke_results.empty else False
)
smoke_top_five = len(smoke_results) <= 5

if not catalog_df.empty:
    reference_sample = catalog_df.iloc[0]
    similarity_results = recommend_similar_to_title(
        reference_sample["title"], kind=reference_sample["kind"], top_n=5
    )
    similarity_smoke_valid = (
        len(similarity_results) <= 5
        and reference_sample["title"] not in set(similarity_results["title"])
        and (similarity_results.empty or np.isfinite(similarity_results["score"].to_numpy()).all())
    )
else:
    similarity_smoke_valid = False
    similarity_results = pd.DataFrame(columns=RESULT_COLUMNS)

phase1_checks = {
    "Base locale détectée": DATABASE_PATH.is_file(),
    "Titres avec métadonnées TMDB actuelles": not catalog_df.empty,
    "Matrice TF-IDF valide": baseline_ready,
    "Top-5 du prompt généré": smoke_top_five and not smoke_results.empty,
    "Scores finis": smoke_scores_finite,
    "Titres sans doublons": smoke_titles_unique,
    "Similarité à un titre testée": similarity_smoke_valid,
}
phase1_technical_passed = all(phase1_checks.values())
phase1_human_review_complete = (
    not review_df.empty
    and review_df["revision_humana"].isin(ALLOWED_LABELS).all()
)

checklist = pd.DataFrame([
    {"Critère": name, "État": "RÉUSSI" if passed else "BLOQUÉ"}
    for name, passed in phase1_checks.items()
])
checklist.loc[len(checklist)] = {
    "Critère": "Évaluation humaine des résultats",
    "État": "RÉUSSIE" if phase1_human_review_complete else "EN ATTENTE",
}
checklist.loc[len(checklist)] = {
    "Critère": "Phase 1 technique",
    "État": "RÉUSSIE" if phase1_technical_passed else "BLOQUÉE",
}
checklist.loc[len(checklist)] = {
    "Critère": "Phase 2 (modèle sémantique et filtres exacts)",
    "État": "BLOQUÉE jusqu’à la revue de cette phase",
}
display(checklist)
print("Résultats du test de requête :")
smoke_display = smoke_results.rename(columns={
    "kind": "Type", "title": "Titre TMDB", "genres": "Genres TMDB (espagnol)",
    "score": "Score TF-IDF", "copies": "Copies locales",
}).copy()
if not smoke_display.empty:
    smoke_display["Type"] = smoke_display["Type"].map({"movies": "Films", "series": "Séries"})
display(smoke_display)
print(f"Référence du test de similarité : {reference_sample['title'] if not catalog_df.empty else 'aucune'}")
similarity_display = similarity_results.rename(columns={
    "kind": "Type", "title": "Titre TMDB", "genres": "Genres TMDB (espagnol)",
    "score": "Score TF-IDF", "copies": "Copies locales",
}).copy()
if not similarity_display.empty:
    similarity_display["Type"] = similarity_display["Type"].map({"movies": "Films", "series": "Séries"})
display(similarity_display)

,Critère,État
0,Base locale détectée,RÉUSSI
1,Titres avec métadonnées TMDB actuelles,RÉUSSI
2,Matrice TF-IDF valide,RÉUSSI
3,Top-5 du prompt généré,RÉUSSI
4,Scores finis,RÉUSSI
5,Titres sans doublons,RÉUSSI
6,Similarité à un titre testée,RÉUSSI
7,Évaluation humaine des résultats,EN ATTENTE
8,Phase 1 technique,RÉUSSIE
9,Phase 2 (modèle sémantique et filtres exacts),BLOQUÉE jusqu’à la revue de cette phase


Résultats du test de requête :


,Type,Titre TMDB,Genres TMDB (espagnol),Score TF-IDF,Copies locales
0,Films,2.0,"Acción, Ciencia ficción, Fantasía",0.276431,1
1,Films,"Invasion, U.S.A.","Drama, Ciencia ficción",0.225438,1
2,Films,Inmortales,"Drama, Ciencia ficción",0.218690,1
3,Films,Sentinel,"Ciencia ficción, Acción",0.206266,1
4,Films,La promesa,"Drama, Ciencia ficción",0.199855,1


Référence du test de similarité : Four Rooms


,Type,Titre TMDB,Genres TMDB (espagnol),Score TF-IDF,Copies locales
0,Films,A Roma con amor,"Comedia, Romance",0.066857,1
1,Films,Padre no hay más que uno,"Comedia, Familia",0.057050,1
2,Films,Cuando Harry encontró a Sally,"Comedia, Romance, Drama",0.056180,1
3,Films,Mickey y las dos brujas,"Animación, Película de TV, Familia",0.054522,1
4,Films,En la cama,"Romance, Drama, Comedia",0.053345,1


## Décision de phase 2 et limites

La revue humaine des 50 candidats est **sautée**, à la demande de l’utilisateur. La qualité/précision subjective n’est donc pas validée; les comparaisons ci-dessus sont exploratoires.

Le modèle FP32 atteint environ **1 367 Mio de RAM** : il ne convient pas à la VM Oracle de 1 Gio. Une variante ONNX INT8 a été testée séparément dans un environnement local Python 3.12 ARM : modèle de **112,9 Mio**, indexation de 8 963 titres en **238,6 s**, environ **803 Mio de RSS** et **3,9 ms** par prompt en moyenne. C’est prometteur mais encore serré; ce n’est pas une validation sur la VM x86_64. Le kernel Python 3.15 actuel ne possède pas de roue ONNX Runtime; utiliser le kernel **Python 3.12 (JaimeTV ONNX)** pour refaire ce test dans le notebook.

**Étape suivante :** exécuter la branche ONNX avec Python 3.12, puis mesurer l’artefact AVX2 sur Oracle avant toute intégration à Flask. Les filtres exacts et la qualité humaine restent à implémenter/évaluer.

In [28]:
phase2_rows = [
    {"Point": "Revue humaine", "État": "Ignorée; la précision subjective n’est pas validée"},
    {"Point": "TF-IDF", "État": "Baseline disponible"},
]
fp32_memory = semantic_metrics.get("RAM après encodage (Mio)") if "semantic_metrics" in globals() else None
phase2_rows.append({
    "Point": "Modèle FP32",
    "État": f"Mesuré à {fp32_memory:.0f} Mio; trop lourd pour la VM de 1 Gio" if fp32_memory else "Mesuré précédemment; trop lourd pour la VM de 1 Gio",
})
phase2_rows.append({
    "Point": "ONNX INT8 local",
    "État": "Testé" if globals().get("onnx_ready", False) else "À lancer avec le kernel Python 3.12 (JaimeTV ONNX)",
})
phase2_rows.append({
    "Point": "Déploiement Oracle x86_64",
    "État": "À valider; aucune modification de l’application n’a été faite",
})
display(pd.DataFrame(phase2_rows))

,Point,État
0,Revue humaine,Ignorée; la précision subjective n’est pas val...
1,TF-IDF,Baseline disponible
2,Modèle FP32,Mesuré à 1415 Mio; trop lourd pour la VM de 1 Gio
3,ONNX INT8 local,À lancer avec le kernel Python 3.12 (JaimeTV O...
4,Déploiement Oracle x86_64,À valider; aucune modification de l’applicatio...
